# 00 -- Setup & Environment Check

This notebook confirms your environment is ready for the hands-on session:
the `image_processing` conda env, the installed `cassa-photometry` package,
the shared Astrometry.net index files (Phase 2), the `solve-field` binary,
and the provided dataset.

> Edit the paths in the config cell below to match the HPC layout you were given.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG
# ============================================================
# One shared module rather than this cell copied into six notebooks, so a
# path is changed once and the notebooks cannot drift apart.
# Override any path with an environment variable; see workshop_config.py.
import importlib, os, sys

_here = os.path.dirname(os.path.abspath('workshop_config.py'))
if _here not in sys.path:
    sys.path.insert(0, _here)

# Reloaded, not merely imported. A kernel that imported workshop_config before
# the file was edited keeps serving the cached module, and the first name added
# since then fails much further down as a bare NameError -- which is exactly how
# `raw_frames()` broke for anyone whose kernel predated it.
import workshop_config
importlib.reload(workshop_config)
from workshop_config import *   # noqa: F403  (RAW_DIR, WORK_DIR, PHASE*_DIR, ...)

require_dataset()   # fails now, with the command that fixes it, not later
os.makedirs(WORK_DIR, exist_ok=True)
show_config()

## 1. The package imports and reports its version

In [ ]:
import cassa_photometry
from cassa_photometry.config import load_config
print('cassa_photometry', cassa_photometry.__version__)
cfg = load_config()
print('default FWHM =', cfg.phase3.fwhm)

## 2. The astrometry index directory is visible
Phase 2 plate-solves against a *local* Astrometry.net index set -- no network
needed. The pipeline looks for `./astrometry_data` relative to the working
directory, which from `workshop/notebooks/` is the wrong place, so
`workshop_config` points `CASSA_ASTROMETRY_INDEX` at the set that ships beside
`workshop/`. An index directory already named in your environment wins.

In [ ]:
from cassa_photometry.config import load_config
idx = load_config().resolve_astrometry_index_dir()
n = len([f for f in os.listdir(idx) if f.startswith('index-')]) if os.path.isdir(idx) else 0
print('Resolved index dir:', idx)
print('Exists:', os.path.isdir(idx), f'({n} index files)')

## 3. `solve-field` (Astrometry.net) is on the PATH

In [ ]:
import shutil
print('solve-field:', shutil.which('solve-field') or 'NOT FOUND -- Phase 2 will fail')

## 4. The dataset is where we expect it

In [ ]:
# The raw tree is whatever the acquisition software wrote. A night arrives as
#     <date>/BIAS|DARK|FLAT|LIGHT/<target>/<frame>.fits
# and a simulated set as one flat directory; `raw_frames()` walks either, which
# is exactly how Phase 1 finds them (cassa_photometry.paths.find_raw_frames).
raw = raw_frames()
print(len(raw), 'raw frames found under', RAW_DIR)

for relative, count in raw_layout().items():
    print(f'  {relative:<40s} {count:4d}')

print()
for f in raw[:5]:
    print(' ', os.path.relpath(f, RAW_DIR))

If all four checks pass, continue to **01 -- The Data Model**.

> Note: Phase 3 (notebook 04) cross-matches against online reference catalogs
> (APASS / Pan-STARRS / SDSS), so that step needs outbound internet on the node.